# Solutions — Logging Like a Production Service

Exercise notebook: `06_files_json_csv/04_logging.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Idempotent logger setup

**Working solution**

In [ ]:
import logging
import sys

def get_logger(name, level="INFO", stream=None):
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level.upper()))
    logger.propagate = False
    if not any(getattr(h, "_lab_handler", False) for h in logger.handlers):
        handler = logging.StreamHandler(stream or sys.stderr)
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
        handler._lab_handler = True
        logger.addHandler(handler)
    return logger

**Explanation**

`logging.getLogger(name)` returns the same object every time (a registry), so handlers accumulate if you add them blindly — the classic 'every log line printed 3 times' bug.

**Complexity:** O(handlers).

**Common mistakes**

- Adding a handler on every call.
- `logging.basicConfig()` in library code — it configures the *root* logger for the whole app.

## Exercise 2 — 🟡 Medium · Rotating file logs

**Working solution**

In [ ]:
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path

def file_logger(name, log_dir, max_bytes=1_000_000, backups=3):
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)
    target = str((log_dir / f"{name}.log").resolve())
    if not any(isinstance(h, RotatingFileHandler) and h.baseFilename == target for h in logger.handlers):
        handler = RotatingFileHandler(target, maxBytes=max_bytes, backupCount=backups, encoding="utf-8")
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
        logger.addHandler(handler)
    return logger

**Explanation**

Rotation caps disk use for long-running jobs. `handler.baseFilename` is absolute, so compare against a resolved path.

**Complexity:** O(1).

**Common mistakes**

- Unbounded log files filling the disk of a long-running worker.

## Exercise 3 — 🟡 Medium · Structured JSON logs

**Working solution**

In [ ]:
import json
import logging
from datetime import datetime, timezone

_STANDARD = set(vars(logging.LogRecord("x", 0, "x", 0, "x", None, None))) | {"message", "asctime"}

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {
            "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        payload.update({k: v for k, v in vars(record).items() if k not in _STANDARD and not k.startswith("_")})
        if record.exc_info:
            payload["exc"] = self.formatException(record.exc_info)
        return json.dumps(payload, default=str)

**Explanation**

CloudWatch Logs Insights, Datadog and ELK can query JSON fields (`filter level = "ERROR" and run_id = "r1"`). `extra=` attaches attributes
to the `LogRecord`; diffing against a blank record's attributes finds them.

**Complexity:** O(fields).

**Common mistakes**

- Using f-strings to embed data in messages — then you can't filter on it.
- Putting extras with names like `message` (clashes with LogRecord).

## Exercise 4 — 🟠 Hard · Context fields for every log line

**Working solution**

In [ ]:
import logging
from contextlib import contextmanager

class _ContextFilter(logging.Filter):
    def __init__(self, fields):
        super().__init__()
        self.fields = fields

    def filter(self, record):
        for key, value in self.fields.items():
            if not hasattr(record, key):
                setattr(record, key, value)
        return True

@contextmanager
def log_context(logger, **fields):
    flt = _ContextFilter(fields)
    logger.filters.insert(0, flt)      # inner (newest) context applies first, so it wins
    try:
        yield logger
    finally:
        logger.removeFilter(flt)

**Explanation**

Filters can *enrich* records, not just drop them. Applying the newest context first and never overwriting existing attributes gives the
"inner wins, explicit extra wins" semantics. In async/multithreaded services you'd use `contextvars` for the same idea.

**Complexity:** O(fields) per record.

**Common mistakes**

- Not removing the filter → fields leak into unrelated later logs.
- Mutating a global dict of 'current context'.

## Challenge — 🔴 Very Hard · Audited job runner with structured logs

**Working solution**

In [ ]:
import time

def run_steps(steps, logger, run_id, clock=time.perf_counter):
    context, results = {}, []
    failed = False
    for name, func in steps:
        if failed:
            results.append({"step": name, "status": "skipped", "duration_ms": None, "rows": None, "error": None})
            continue
        logger.info("step started", extra={"run_id": run_id, "step": name})
        start = clock()
        try:
            rows = func(context)
        except Exception as exc:
            duration = round((clock() - start) * 1000)
            logger.error("step failed", exc_info=True, extra={"run_id": run_id, "step": name, "duration_ms": duration})
            results.append({"step": name, "status": "failed", "duration_ms": duration, "rows": None,
                            "error": f"{type(exc).__name__}: {exc}"})
            failed = True
            continue
        duration = round((clock() - start) * 1000)
        logger.info("step finished", extra={"run_id": run_id, "step": name, "duration_ms": duration, "rows": rows})
        results.append({"step": name, "status": "success", "duration_ms": duration, "rows": rows, "error": None})
    return {"run_id": run_id, "status": "FAILED" if failed else "SUCCESS", "steps": results,
            "total_rows": sum(r["rows"] or 0 for r in results)}

**Explanation**

This is the skeleton of every orchestrated job: per-step timing, structured logs you can query by `run_id`, a machine-readable run summary
(the "audit record" you'd write to DynamoDB/a control table), and fail-fast with explicit `skipped` status. You'll reuse it in the projects.

**Complexity:** O(steps).

**Common mistakes**

- Raising out of the runner and losing the summary.
- Timing with `time.time()` (wall clock can jump).